# GPT od podstaw

Warsztaty w ramach Funduszu Zdolni (30 kwietnia - 2 maja 2026).

## Notebook 3: Prosta sieć neuronowa w PyTorch

W tym zeszycie przechodzimy z liczenia wystąpień na uczenie maszynowe. Zbudujemy prostą sieć neuronową, która:
1. Zamienia litery na liczby, np. literę `"A"` na liczbę `13`.
2. Każdej liczbie przypisuje listę liczb zmiennoprzecinkowych (np. `[0.1, -0.5, 0.8]`), co pozwala sieci płynnie operować na znakach (to tzw. **embeddings**).
3. Patrzy na kilka poprzednich znaków, np. widząc `"Lit"`, próbuje zgadnąć, że następną literą będzie `"w"` (od słowa `"Litwo"`).

Do śledzenia treningu uzywamy [livelossplot](https://github.com/stared/livelossplot).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from livelossplot import PlotLosses

### 1. Słownik i zamiana liter na liczby

Sieci neuronowe nie rozumieją tekstu, umieją tylko mnożyć i dodawać liczby. Dlatego najpierw musimy stworzyć słownik, który np. literze `"a"` przypisze liczbę `0`, `"b"` liczbę `1`, itd.

In [ ]:
# Wczytujemy tekst
with open("data/pan-tadeusz.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Tworzymy posortowaną listę wszystkich unikalnych znaków w tekście
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Rozmiar słownika (unikalne znaki): {vocab_size}")

# Słowniki do tłumaczenia w obie strony
char2id = {c: i for i, c in enumerate(chars)}
id2char = {i: c for i, c in enumerate(chars)}

print("A ->", char2id.get("A", "Brak w tekście"))
print(char2id.get("A", 0), "->", id2char[char2id.get("A", 0)])

### 2. Przygotowanie przykładów do nauki: Kontekst i Cel

Sieć uczy się na przykładach. Jeśli mamy tekst `"Litwo"`, i patrzymy na 3 litery wstecz (nasz kontekst), naszymi przykładami do nauki będą:
* Widząc `"Lit"` (kontekst), zgadnij `"w"` (cel).
* Widząc `"itw"` (kontekst), zgadnij `"o"` (cel).

W kodzie kontekst często nazywa się `X`, a cel `Y`.

**Zbiór treningowy i testowy**
Aby sprawdzić, czy sieć faktycznie się uczy, a nie tylko uczy się tekstu na pamięć (tzw. *overfitting*), podzielimy tekst:
* **75% początkowego tekstu** posłuży do nauki (zbiór treningowy).
* **25% końcowego tekstu** posłuży do sprawdzania wiedzy (zbiór testowy).

In [ ]:
context_size = 3  # Patrzymy na 3 znaki wstecz

# Dzielimy tekst na część do nauki (75%) i do testów (25%)
split_idx = int(len(text) * 0.75)
train_text = text[:split_idx]
test_text = text[split_idx:]

def zbuduj_zbior(fragment_tekstu):
    X, Y = [], []
    for i in range(len(fragment_tekstu) - context_size):
        context = fragment_tekstu[i:i+context_size]
        target = fragment_tekstu[i+context_size]
        X.append([char2id[c] for c in context])
        Y.append(char2id[target])
    return torch.tensor(X), torch.tensor(Y)

X_train, Y_train = zbuduj_zbior(train_text)
X_test, Y_test = zbuduj_zbior(test_text)

print(f"Zbiór treningowy: {X_train.shape[0]} przykładów")
print(f"Zbiór testowy: {X_test.shape[0]} przykładów")
print("\nPrzykład z treningowego:")
print(f"Kontekst (X): {X_train[0].tolist()} -> '{''.join([id2char[i] for i in X_train[0].tolist()])}'")
print(f"Cel (Y): {Y_train[0].item()} -> '{id2char[Y_train[0].item()]}'")

### 3. Budowa sieci neuronowej

Zbudujemy dwa modele, żeby zobaczyć różnicę:

**1. Regresja Logistyczna (LogisticRegression)**:
Najprostszy model. Zamienia znaki na wektory (Embedding), spłaszcza je i od razu za pomocą jednej warstwy liniowej (Linear) ocenia, jaki będzie następny znak.

**2. Wielowarstwowy Perceptron (MultiLayerPerceptron)**:
Model z dodatkową warstwą ukrytą, która pozwala mu uczyć się bardziej skomplikowanych zależności.
1. **Embedding**: Zamienia pojedynczą liczbę (np. `13` dla `"A"`) na listę 10 liczb zmiennoprzecinkowych.
2. **Warstwa ukryta**: Bierze wektory z 3 poprzednich liter (razem 30 liczb), mnoży je przez wagi, dodaje i przepuszcza przez funkcję aktywacji **ReLU** (która po prostu zamienia liczby ujemne na zero), żeby wyciągnąć wnioski z tego ciągu znaków.
3. **Warstwa wyjściowa**: Na podstawie tych wniosków, wypluwa ocenę dla każdej możliwej litery w słowniku.

In [ ]:
class LogisticRegression(nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int, context_size: int):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embedding_dim)
        self.linear = nn.Linear(context_size * embedding_dim, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        emb = self.emb(x)
        emb_flat = emb.view(emb.shape[0], -1)
        logits = self.linear(emb_flat)
        return logits

class MultiLayerPerceptron(nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int, context_size: int, hidden_dim: int):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embedding_dim)
        self.linear1 = nn.Linear(context_size * embedding_dim, hidden_dim)
        self.linear2 = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        emb = self.emb(x)
        emb_flat = emb.view(emb.shape[0], -1)
        # Funkcja aktywacji ReLU (Rectified Linear Unit)
        hidden = torch.relu(self.linear1(emb_flat))
        logits = self.linear2(hidden)
        return logits

# Tworzymy instancję bardziej zaawansowanego modelu (MLP)
# Możesz zmienić na model = LogisticRegression(vocab_size, embedding_dim, context_size) by zobaczyć różnicę!
embedding_dim = 10
hidden_dim = 64
model = MultiLayerPerceptron(vocab_size, embedding_dim, context_size, hidden_dim)
print(model)

### 4. Nauka na błędach (Trenowanie)

Sieć na początku zgaduje losowo. Funkcja straty oblicza, jak bardzo się pomyliła (np. dała wysoką ocenę literze `"z"`, a poprawną było `"w"`). Następnie optymalizator (tutaj algorytm o nazwie Adam) lekko poprawia wagi w sieci, żeby następnym razem błąd był mniejszy.

Użyjemy biblioteki `livelossplot`, aby na żywo rysować wykres błędu (straty) dla zbioru treningowego i testowego.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

batch_size = 1024
epochs = 1000  # Zmniejszyłem do 1000, bo z wykresem widać szybciej

liveloss = PlotLosses()

print("Rozpoczynamy trening...")
for epoch in range(epochs):
    logs = {}
    
    # --- TRENING ---
    model.train()
    idx = torch.randint(0, X_train.shape[0], (batch_size,))
    xb, yb = X_train[idx], Y_train[idx]

    logits = model(xb)
    loss = criterion(logits, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    logs['loss'] = loss.item()

    # --- TESTOWANIE (co 10 epok) ---
    if epoch % 10 == 0 or epoch == epochs - 1:
        model.eval()
        with torch.no_grad():
            # Losujemy paczkę ze zbioru testowego
            idx_test = torch.randint(0, X_test.shape[0], (batch_size,))
            xb_test, yb_test = X_test[idx_test], Y_test[idx_test]
            
            logits_test = model(xb_test)
            loss_test = criterion(logits_test, yb_test)
            logs['val_loss'] = loss_test.item()
        
        # Aktualizujemy wykres
        liveloss.update(logs)
        liveloss.send()

print("Trening zakończony!")

### 5. Generowanie tekstu

Podajemy sieci np. `"Lit"`. Sieć ocenia wszystkie litery, my zamieniamy te oceny na prawdopodobieństwa i losujemy kolejną literę (np. wypada `"w"`). Następnie przesuwamy okno: podajemy `"itw"` i prosimy o kolejną literę.

In [ ]:
def generuj_tekst(model: nn.Module, start_str: str, dlugosc: int = 200) -> str:
    # Upewniamy się, że ciąg startowy ma odpowiednią długość
    if len(start_str) < context_size:
        start_str = start_str.rjust(context_size, " ")
        
    # Bierzemy tylko ostatnie `context_size` znaków
    context = [char2id[c] for c in start_str[-context_size:]]
    wynik = start_str
    
    for _ in range(dlugosc):
        # Zmieniamy na tensor o kształcie (1, context_size)
        x = torch.tensor([context])
        
        # Przewidujemy
        with torch.no_grad():
            logits = model(x)
            
        # Zamieniamy logits na prawdopodobieństwa (Softmax)
        probs = F.softmax(logits, dim=1)
        
        # Losujemy następny znak na podstawie prawdopodobieństw
        next_id = torch.multinomial(probs, num_samples=1).item()
        
        # Dodajemy do wyniku
        wynik += id2char[next_id]
        
        # Przesuwamy okno kontekstu
        context = context[1:] + [next_id]
        
    return wynik

print("Wygenerowany tekst:\n")
print(generuj_tekst(model, "Lit", dlugosc=300))